In [2]:
"""
PIPELINE STAGE: Capacity Table Extraction from Monthly Energy Reports

INPUT: raw_data/energy_reports/{2021-2025}/*.docx
OUTPUT: processed_data/steps/01_Capacity.xlsx
LIBRARIES: os, zipfile, xml.etree.ElementTree, pandas

1. OBJECTIVE:
    Extract provincial renewable energy installed capacity (capacity) tables from monthly
    Word energy reports and consolidate them into a single master dataset. The workflow
    reads DOCX files at XML level, automatically identifies target capacity tables, appends
    temporal metadata (Year, Month), and exports a unified Excel file for downstream
    energy analytics and machine learning studies.

2. DOCUMENT PARSING & XML EXTRACTION:
    A. Direct DOCX Processing:
       - Treat each .docx file as a ZIP archive.
       - Read "word/document.xml" directly without using python-docx.
       - Parse XML structure using ElementTree.

    B. Word Table Extraction:
       - Locate all Word tables (<w:tbl>) within the XML document.
       - Extract all rows (<w:tr>) and cells (<w:tc>).
       - Merge text fragments (<w:t>) within each cell.
       - Remove empty rows.
       - Preserve tabular structure during extraction.

    C. Table Normalization:
       - Determine the maximum column count in each table.
       - Pad shorter rows with empty strings to maintain structural consistency.
       - Convert normalized tables into pandas DataFrames.
       - Ignore tables containing fewer than two rows.

3. TARGET TABLE IDENTIFICATION:
    A. Content-Based Detection:
       - Analyze both column headers and the first data row.
       - Convert all text to lowercase.
       - Normalize Turkish character artifacts:
           replace("i̇", "i")

    B. Required Keywords:
       A table is considered a valid installed-capacity table only if all
       of the following keywords are present:

           * iller
           * biyokütle
           * güneş
           * rüzgar

    C. False Table Elimination:
       - Skip all tables that do not satisfy the keyword requirements.
       - Retain only capacity-related tables.

4. HEADER CORRECTION & STRUCTURAL REPAIR:
    A. Header Validation:
       - Verify whether the extracted column names contain the keyword "iller".

    B. Header Recovery:
       - If "iller" is not found in the DataFrame columns:
           * Promote the first data row to header.
           * Remove the promoted row from the dataset.
           * Reset row indexing.

    C. Schema Preservation:
       - Maintain the original capacity table structure exactly as reported.

5. TEMPORAL METADATA EXTRACTION:
    A. Filename Parsing:
       Extract metadata from filenames formatted as:

           january_2023.docx
           february_2024.docx
           ...

    B. Month Transformation:
       - Use the filename month string directly.
       - Capitalize the first character.

       Examples:
           january → January
           february → February

    C. Year Assignment:
       - Read year information from filename.
       - If unavailable, use the current folder name.

6. TABLE SELECTION RULE:
    A. Multiple Matching Tables:
       - Some reports may contain more than one valid table.

    B. Extraction Strategy:
       - Select only the first valid capacity table.
       - Ignore subsequent matching tables.

    C. Metadata Enrichment:
       Append:

           Year
           Month

       columns to the extracted table before storage.

7. MULTI-YEAR CONSOLIDATION:
    A. Target Years:
           2021
           2022
           2023
           2024
           2025

    B. Directory Traversal:
       - Scan each yearly folder recursively.
       - Process all DOCX files.

    C. Temporary File Exclusion:
       - Ignore files beginning with "~".
       - Ignore non-DOCX files.

    D. Master Dataset Assembly:
       - Store each extracted capacity table in memory.
       - Concatenate all monthly datasets into a single DataFrame.

8. OUTPUT GENERATION:
    A. Final Dataset:
       Merge all extracted capacity tables using:

           pd.concat(..., ignore_index=True)

    B. Export Format:
       Save the master dataset as:

           processed_data/steps/01_Capacity.xlsx

    C. Export Settings:
       - Excel format (.xlsx)
       - No index column

9. PROCESS LOGGING & MONITORING:
    A. Runtime Logging:
       Display:
           - Current year folder
           - Current file name
           - Extracted Year
           - Extracted Month

    B. Warning Messages:
       Report files where no valid capacity table is detected.

    C. Completion Messages:
       Report successful creation and save location of the final master dataset.

10. EXPECTED OUTPUT DATASET:
    The resulting Excel file contains:

        - Original installed-capacity table columns
        - Provincial capacity values
        - Renewable energy source capacities
        - Year
        - Month

    The dataset serves as the raw installed-capacity master table for
    subsequent cleaning, standardization, feature engineering,
    energy analytics, and machine learning workflows.
"""
import os
import zipfile
import xml.etree.ElementTree as ET
import pandas as pd

class CapacityExtractor:
    def __init__(self, base_dir):
        self.raw_root = os.path.join(base_dir, "raw_data", "energy_reports")
        self.final_dir = os.path.join(base_dir, "processed_data", "steps")
        self.ns = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}
        
        os.makedirs(self.final_dir, exist_ok=True)

    def _parse_xml_table(self, table_el):
        rows = []
        for row_el in table_el.findall('.//w:tr', self.ns):
            cells = []
            for cell_el in row_el.findall('.//w:tc', self.ns):
                cell_text = "".join([t.text for t in cell_el.findall('.//w:t', self.ns) if t.text])
                cells.append(cell_text.strip())
            
            if any(cells):
                rows.append(cells)
        
        if len(rows) < 2:
            return None
            
        max_cols = max(len(row) for row in rows)
        normalized_rows = [row + [''] * (max_cols - len(row)) for row in rows]
        return pd.DataFrame(normalized_rows[1:], columns=normalized_rows[0])

    def _is_target_table(self, df):
        if df is None or len(df) < 1:
            return False
            
        cols_text = " ".join([str(c) for c in df.columns]).lower().replace("i̇", "i")
        row0_text = " ".join([str(c) for c in df.iloc[0]]).lower().replace("i̇", "i")
        combined = cols_text + " " + row0_text
        required_words = ["iller", "biyokütle", "güneş", "rüzgar"]
        
        return all(w in combined for w in required_words)

    def extract_tables(self, file_path):
        valid_tables = []
        with zipfile.ZipFile(file_path, 'r') as docx_zip:
            xml_content = docx_zip.read('word/document.xml')
            root = ET.fromstring(xml_content)
            
            for table_el in root.findall('.//w:tbl', self.ns):
                df = self._parse_xml_table(table_el)
                
                if self._is_target_table(df):
                    cols_lower = " ".join([str(c) for c in df.columns]).lower().replace("i̇", "i")
                    if "iller" not in cols_lower:
                        df.columns = df.iloc[0]
                        df = df.drop(0).reset_index(drop=True)
                    valid_tables.append(df)
                    
        return valid_tables

    def run_for_file(self, year_folder, docx_name, master_capacity_list):
        file_path = os.path.join(self.raw_root, year_folder, docx_name)
        if not os.path.exists(file_path):
            return

        base_name = os.path.splitext(docx_name)[0]
        parts = base_name.split('_')
        
        # Ayı doğrudan isim olarak al ve baş harfini büyüt (Örn: january -> January)
        month_val = parts[0].capitalize() if len(parts) > 0 else ""
        year_val = parts[1] if len(parts) > 1 else year_folder

        print(f">>> Hafızaya Alınıyor (Capacity): {year_folder}/{docx_name} -> (Yıl: {year_val}, Ay: {month_val})")
        valid_tables = self.extract_tables(file_path)
        
        # SADECE 1. Tablo (Kurulu Güç) - HAM Veri
        if len(valid_tables) >= 1:
            df_cap = valid_tables[0].copy()
            df_cap["Yıl"] = year_val
            df_cap["Ay"] = month_val
            master_capacity_list.append(df_cap)
        else:
            print(f"     [X] Uyarı: {docx_name} belgesinde uygun tablo bulunamadı.")

    def process_and_combine_capacity(self):
        print("\n🚀 SADECE CAPACITY (KURULU GÜÇ) İŞLEMİ BAŞLIYOR...")
        
        master_capacity_dfs = []
        years = ["2021", "2022", "2023", "2024", "2025"]
        
        for year in years:
            year_dir = os.path.join(self.raw_root, year)
            if not os.path.exists(year_dir):
                continue
                
            for file_name in os.listdir(year_dir):
                if file_name.endswith(".docx") and not file_name.startswith("~"):
                    self.run_for_file(year, file_name, master_capacity_dfs)
                    
        print("\n📊 Bütün aylar tarandı. Capacity (Kurulu Güç) Master dosyası oluşturuluyor...")
        
        if master_capacity_dfs:
            combined_capacity = pd.concat(master_capacity_dfs, ignore_index=True)
            final_cap_path = os.path.join(self.final_dir, "01_Capacity.xlsx")
            combined_capacity.to_excel(final_cap_path, index=False)
            print(f" [✓] İŞLEM TAMAM -> Kurulu Güç Ham Master Tablosu Kaydedildi: {final_cap_path}")
        else:
            print(" [X] Hata: Birleştirilecek hiçbir Kurulu Güç tablosu bulunamadı.")


# --- ÇALIŞTIRMA ---
if __name__ == "__main__":
    BASE_PROJECT_DIR = r"C:\Users\ismailgulsoy\dergi2"
    
    cap_extractor = CapacityExtractor(BASE_PROJECT_DIR)
    cap_extractor.process_and_combine_capacity()

SyntaxError: invalid decimal literal (3164867716.py, line 4)